# Standalone MuseTalk + LivePortrait Talking-Head Generator

This notebook installs and runs a **dual-stage standalone pipeline** on Google Colab's free GPU tier to generate YouTube-quality talking head videos:
1. **LivePortrait Stage**: Animates your static avatar image (PNG/JPG) using a natural talking-head driving template video (blinking, breathing, head tilting).
2. **MuseTalk Stage**: Lip-syncs the animated mouth area perfectly to match your voiceover audio (WAV).
3. **FFmpeg Stage**: Merges the voiceover audio back into the final video.

*If you upload a video (.mp4) as the avatar instead of a static image, the notebook will automatically skip the LivePortrait animation stage and run MuseTalk lip-sync directly on it.*

### Setup Instructions:
1. Ensure your runtime is running on a GPU: Go to **Runtime > Change runtime type**, select **T4 GPU** (or any other GPU option), and click Save.
2. Run **Cell 1 (Setup Environment)**.
3. Run **Cell 2 (Download Model Weights)**.
4. Upload your avatar (PNG/JPG or MP4) and voiceover audio (WAV) directly into Google Colab's file explorer sidebar (click the folder icon on the left).
5. Update file paths in **Cell 3 (Generate Talking Head Video)** and run it!


In [ ]:
# 1. Install Python 3.10 and development dependencies
!sudo add-apt-repository ppa:deadsnakes/ppa -y
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-dev python3.10-venv ffmpeg -y

# 2. Create virtual environment running Python 3.10
!python3.10 -m venv /content/venv310

# 3. Upgrade pip and setuptools inside the virtual environment
!/content/venv310/bin/pip install --upgrade pip setuptools

# 4. Install PyTorch 2.1.2 + CUDA 12.1 inside the virtual environment (stable for MuseTalk & LivePortrait)
!/content/venv310/bin/pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# 5. Clone MuseTalk and LivePortrait repositories (if not already cloned)
import os
if not os.path.exists("/content/MuseTalk"):
    !git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk
if not os.path.exists("/content/LivePortrait"):
    !git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait

# 6. Install build tools (cython and numpy) first so C extensions can compile successfully inside venv
!/content/venv310/bin/pip install cython numpy

# 7. Clean all requirements files to prevent pip from overwriting GPU PyTorch or ONNX
import os
def clean_reqs(filepath):
    if not os.path.exists(filepath): return
    with open(filepath, "r") as f: lines = f.readlines()
    cleaned = [l for l in lines if not any(p in l.lower() for p in ["torch", "torchvision", "torchaudio", "onnxruntime"])]
    with open(filepath, "w") as f: f.writelines(cleaned)
    print(f"Cleaned {filepath}")

clean_reqs("/content/MuseTalk/requirements.txt")
clean_reqs("/content/LivePortrait/requirements.txt")
if os.path.exists("/content/LivePortrait/requirements_base.txt"):
    clean_reqs("/content/LivePortrait/requirements_base.txt")

# Install requirements inside venv (PyTorch and ONNX are protected)
!/content/venv310/bin/pip install -r /content/MuseTalk/requirements.txt
!/content/venv310/bin/pip install -r /content/LivePortrait/requirements.txt

# 8. Install build-dependent packages inside venv
!/content/venv310/bin/pip install xtcocotools
!/content/venv310/bin/pip install chumpy --no-build-isolation

# 9. Install OpenMMLab packages directly inside the venv using pip
!/content/venv310/bin/pip install mmengine==0.10.4 mmdet==3.3.0 mmpose==1.3.2
!/content/venv310/bin/pip install mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1/index.html

# 10. Install extra dependencies (fastapi, gfpgan, tyro, etc.) inside venv
!/content/venv310/bin/pip install setuptools==59.6.0 fastapi uvicorn[standard] python-multipart pydantic huggingface_hub gfpgan tyro
!/content/venv310/bin/pip uninstall -y onnxruntime onnxruntime-gpu
!/content/venv310/bin/pip install -U insightface onnxruntime-gpu==1.17.1

# 11. Verify that all critical libraries are installed and can import successfully inside the venv
print("\n--- VERIFYING ENVIRONMENT STACK INSIDE VIRTUAL ENVIRONMENT ---")
verification_script = """
import sys
success = True
for module in ['torch', 'mmpose', 'insightface', 'tyro', 'gfpgan', 'chumpy']:
    try:
        __import__(module)
        print(f'  [✓] {module} imported successfully.')
    except ImportError as e:
        print(f'  [✗] Failed to import {module}: {e}')
        success = False

import torch
cuda_ok = torch.cuda.is_available()
print(f'  [✓] GPU Available: {cuda_ok}')
if not cuda_ok:
    success = False

if not success:
    sys.exit(1)
print('\\n🎉 ENVIRONMENT IS 100% READY!')
"""
with open("/content/verify_env.py", "w") as f:
    f.write(verification_script)
!/content/venv310/bin/python /content/verify_env.py

print("\n✅ All repositories and environment dependencies set up successfully in Python 3.10 virtual environment!")


In [ ]:
import os
import shutil
from huggingface_hub import hf_hub_download

print("1. Downloading MuseTalk model weights...")

# Define MuseTalk files and repositories
downloads = [
    ("TMElyralab/MuseTalk", "musetalk/musetalk.json", "musetalk/musetalk.json"),
    ("TMElyralab/MuseTalk", "musetalk/musetalk.json", "musetalk/config.json"),
    ("TMElyralab/MuseTalk", "musetalk/pytorch_model.bin", "musetalk/pytorch_model.bin"),
    ("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", "musetalkV15/musetalk.json"),
    ("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", "musetalkV15/config.json"),
    ("TMElyralab/MuseTalk", "musetalkV15/unet.pth", "musetalkV15/unet.pth"),
    ("yzd-v/DWPose", "dw-ll_ucoco_384.pth", "dwpose/dw-ll_ucoco_384.pth"),
    ("ManyOtherFunctions/face-parse-bisent", "79999_iter.pth", "face-parse-bisent/79999_iter.pth"),
    ("ManyOtherFunctions/face-parse-bisent", "resnet18-5c106cde.pth", "face-parse-bisent/resnet18-5c106cde.pth"),
    ("stabilityai/sd-vae-ft-mse", "config.json", "sd-vae-ft-mse/config.json"),
    ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", "sd-vae-ft-mse/diffusion_pytorch_model.bin"),
    ("stabilityai/sd-vae-ft-mse", "config.json", "sd-vae/config.json"),
    ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", "sd-vae/diffusion_pytorch_model.bin"),
    ("openai/whisper-tiny", "config.json", "whisper/config.json"),
    ("openai/whisper-tiny", "preprocessor_config.json", "whisper/preprocessor_config.json"),
    ("openai/whisper-tiny", "pytorch_model.bin", "whisper/pytorch_model.bin")
]

models_dir = "/content/MuseTalk/models"
os.makedirs(models_dir, exist_ok=True)

for repo_id, filename, target_rel_path in downloads:
    target_path = os.path.join(models_dir, target_rel_path)
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    if not os.path.exists(target_path):
        print(f"Downloading {filename} from {repo_id}...")
        downloaded_file = hf_hub_download(repo_id=repo_id, filename=filename)
        shutil.copy(downloaded_file, target_path)

print("\n2. Downloading LivePortrait model weights programmatically...")
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="KwaiVGI/LivePortrait",
    local_dir="/content/LivePortrait/pretrained_weights",
    ignore_patterns=["*.git*", "*README.md*", "*docs/*"]
)

print("\n3. Downloading GFPGAN model weights...")
gfpgan_url = "https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth"
gfpgan_path = "/content/GFPGANv1.4.pth"
if not os.path.exists(gfpgan_path):
    print("Downloading GFPGANv1.4.pth...")
    import urllib.request
    urllib.request.urlretrieve(gfpgan_url, gfpgan_path)

print("\n✅ All weights downloaded and organized successfully!")


## Generate Talking Head Video

1. **Upload files**: Click the **folder icon** on the left sidebar of Google Colab and upload your avatar image (PNG/JPG) or video (MP4) along with your voiceover audio (WAV) directly into the `/content/` directory.
2. **Configure paths**: Update `avatar_path`, `audio_path`, and `driving_video` in the cell below.

### 💡 Tips for Best Results:
*   **Quality (Sharp Lips & Teeth)**: Set `enable_gfpgan = True` in the configuration. MuseTalk operates in a 256x256 bounding box which can make the mouth and teeth look blurry. Enabling GFPGAN v1.4 will perform face restoration frame-by-frame, removing all blur, reconstructing sharp teeth, and upscaling the output by 2x for professional YouTube quality.
*   **Natural Head Motion**: The default driving template `d0.mp4` is very short (~5s) and only blinks/smiles. If your audio is long, this motion loops repetitively. We recommend using `d13.mp4` (in the `assets/examples/driving/` folder) which is a natural talking video clip.
*   **Expression & Word Emphasis (Self-Driving)**: Since LivePortrait is video-driven and MuseTalk is audio-driven, a generic driving video will *not* align expressions (nods, raised eyebrows) with emphasized words. **To fix this, record a quick driving video of yourself (or a reference actor) speaking the words with the desired expressions and head gestures.** Upload it and set it as your `driving_video`. LivePortrait will map your expressions and head motions directly onto the avatar, and MuseTalk will sync the mouth to the audio. This yields perfect expression-to-speech alignment!


In [ ]:
avatar_path = "/content/avatar_fitness.png"  # Path to your uploaded avatar image (PNG/JPG) or video (MP4)
audio_path = "/content/voiceover_trimmed_10s.wav"    # Path to your uploaded voiceover audio (WAV)
driving_video = "/content/LivePortrait/assets/examples/driving/d13.mp4" # Driving template video (d13.mp4 is recommended for talking head)

bbox_shift = -7  # Bounding box vertical shift (-7 for fitness girl, -3 for others)
use_float16 = True

# Quality restoration parameters
enable_gfpgan = True  # Set to True to remove blur on lips/mouth and restore proper teeth
upscale_factor = 2    # Upscale factor for output video (1 for no upscale, 2 for 2x resolution)

# LivePortrait parameters
flag_crop_driving_video = True  # Set to True if your driving video is not square (1:1) or centered on face

import os
import subprocess
import shutil
import glob
from pathlib import Path

# Verify inputs
if not os.path.exists(avatar_path):
    raise FileNotFoundError(f"Avatar file not found at {avatar_path}. Please upload it using the left sidebar.")
if not os.path.exists(audio_path):
    raise FileNotFoundError(f"Audio file not found at {audio_path}. Please upload it using the left sidebar.")

ext = os.path.splitext(avatar_path)[1].lower()
is_static_image = ext in ['.png', '.jpg', '.jpeg', '.webp']

print(f"Detected avatar type: {'Static Image' if is_static_image else 'Video Clip'}")

work_dir = Path("/content/standalone_job")
work_dir.mkdir(parents=True, exist_ok=True)

try:
    # ─── Stage 0: Driving Video Standardization (Enforce 25 FPS and MP4 format) ───
    # High frame rates (like 59/60 FPS) cause Out of Memory crashes (exit code -9) on Colab.
    # We standardize the driving video to 25 FPS and a standard H.264 MP4 container.
    temp_driving = "/content/temp_driving_standardized.mp4"
    print(f"Standardizing driving video to 25 FPS and compatible container: {temp_driving}...")
    conv_res = subprocess.run([
        "ffmpeg", "-y",
        "-i", driving_video,
        "-r", "25",                     # Enforce 25 FPS
        "-c:v", "libx264",              # Standard H.264 video codec
        "-preset", "superfast",
        "-crf", "23",
        "-c:a", "aac",                  # Standard AAC audio
        temp_driving
    ], capture_output=True, text=True)
    if conv_res.returncode == 0 and os.path.exists(temp_driving):
        driving_video = temp_driving
        print("✅ Driving video standardized successfully (25 FPS).")
    else:
        print(f"Warning: Standardizing failed, using original driving file. Stderr: {conv_res.stderr}")
            
    if is_static_image:
        # ─── Stage 1: LivePortrait (Animate Head) ───
        print("\n--- STAGE 1: Animating Face using LivePortrait ---")
        lp_out_dir = work_dir / "liveportrait"
        
        lp_cmd = [
            "/content/venv310/bin/python", "inference.py",
            "-s", avatar_path,
            "-d", driving_video,
            "-o", str(lp_out_dir)
        ]
        if flag_crop_driving_video:
            lp_cmd.append("--flag_crop_driving_video")
            
        print(f"Running LivePortrait command: {' '.join(lp_cmd)}")
        
        env = os.environ.copy()
        env["MPLBACKEND"] = "agg"
        
        lp_res = subprocess.run(lp_cmd, cwd="/content/LivePortrait", env=env, capture_output=True, text=True)
        if lp_res.returncode != 0:
            print("LivePortrait Stdout:", lp_res.stdout)
            print("LivePortrait Stderr:", lp_res.stderr)
            raise RuntimeError(f"LivePortrait animation failed with exit code {lp_res.returncode}")
            
        lp_mp4_files = glob.glob(str(lp_out_dir / "**/*.mp4"), recursive=True)
        if not lp_mp4_files:
            print("LivePortrait Stdout:", lp_res.stdout)
            print("LivePortrait Stderr:", lp_res.stderr)
            raise RuntimeError("LivePortrait completed but no output MP4 found.")
            
        animated_video_path = max(lp_mp4_files, key=os.path.getmtime)
        print(f"Successfully generated animated face video: {animated_video_path}")
        
    else:
        # Skip LivePortrait if input is already a video
        animated_video_path = avatar_path
        print("\n--- STAGE 1 SKIPPED: Using source video directly ---")

    # ─── Stage 2: MuseTalk (Lip-Sync) ───
    print("\n--- STAGE 2: Lip-Syncing with MuseTalk ---")
    
    # 1. Write config yaml
    config_path = work_dir / "musetalk_config.yaml"
    config_content = f"task_0:\n  video_path: \"{animated_video_path}\"\n  audio_path: \"{audio_path}\"\n"
    with open(config_path, "w") as f:
        f.write(config_content)
        
    result_dir = work_dir / "musetalk_result"
    result_dir.mkdir(parents=True, exist_ok=True)
    
    # 2. Run MuseTalk inference using the Python 3.10 virtual environment
    cmd = [
        "/content/venv310/bin/python", "-m", "scripts.inference",
        "--inference_config", str(config_path),
        "--result_dir", str(result_dir),
        "--version", "v15",
        "--bbox_shift", str(bbox_shift),
    ]
    if use_float16:
        cmd.append("--use_float16")
        
    print(f"Running MuseTalk command: {' '.join(cmd)}")
    env = os.environ.copy()
    env["MPLBACKEND"] = "agg"
    
    res = subprocess.run(cmd, cwd="/content/MuseTalk", env=env, capture_output=True, text=True)
    
    # Print stdout/stderr logs for transparency
    print("\n=== MUSETALK STDOUT ===")
    print(res.stdout)
    print("=======================")
    print("\n=== MUSETALK STDERR ===")
    print(res.stderr)
    print("=======================")
    
    if res.returncode != 0:
        raise RuntimeError(f"MuseTalk failed with exit code {res.returncode}")
        
    # Find output video
    mp4_files = list(result_dir.rglob("*.mp4"))
    if not mp4_files:
        raise RuntimeError("No output MP4 found.")
    output_mp4 = str(max(mp4_files, key=os.path.getmtime))
    
    # Define primary video input for FFmpeg
    final_video_input = output_mp4

    # ─── Stage 2.5: Face Enhancement (GFPGAN) ───
    if enable_gfpgan:
        print("\n--- STAGE 2.5: Enhancing Face Quality with GFPGAN ---")
        
        # Write the face enhancement script dynamically
        enhance_lines = [
            "import os",
            "import sys",
            "import cv2",
            "from tqdm import tqdm",
            "",
            "try:",
            "    import torchvision.transforms.functional as TF",
            "    sys.modules['torchvision.transforms.functional_tensor'] = TF",
            "except Exception as e:",
            "    pass",
            "",
            "from gfpgan import GFPGANer",
            "",
            "def main():",
            "    input_path = sys.argv[1]",
            "    output_path = sys.argv[2]",
            "    model_path = sys.argv[3]",
            "    upscale = int(sys.argv[4]) if len(sys.argv) > 4 else 2",
            "",
            "    import torch",
            "    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'",
            "    print(f\'Initializing GFPGAN on device: {device} (model_path={model_path}, upscale={upscale})...\')",
            "    restorer = GFPGANer(",
            "        model_path=model_path,",
            "        upscale=upscale,",
            "        arch=\'clean\',",
            "        channel_multiplier=2,",
            "        bg_upsampler=None,",
            "        device=device",
            "    )",
            "",
            "    cap = cv2.VideoCapture(input_path)",
            "    fps = cap.get(cv2.CAP_PROP_FPS)",
            "    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) * upscale",
            "    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) * upscale",
            "    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))",
            "",
            "    print(f\'Processing {total_frames} frames...\')",
            "    fourcc = cv2.VideoWriter_fourcc(*\'mp4v\')",
            "    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))",
            "",
            "    for _ in tqdm(range(total_frames), desc=\'GFPGAN Enhancement\'):",
            "        ret, frame = cap.read()",
            "        if not ret:",
            "            break",
            "        _, _, restored_img = restorer.enhance(frame, has_aligned=False, only_center_face=False, paste_back=True)",
            "        out.write(restored_img)",
            "",
            "    cap.release()",
            "    out.release()",
            "    print(\'GFPGAN processing complete!\')",
            "",
            "if __name__ == \'__main__\':",
            "    main()"
        ]
        
        with open("/content/enhance.py", "w") as f:
            f.write("\n".join(enhance_lines))
            
        enhanced_output_path = str(work_dir / "enhanced_output.mp4")
        
        enhance_cmd = [
            "/content/venv310/bin/python", "/content/enhance.py",
            output_mp4,
            enhanced_output_path,
            "/content/GFPGANv1.4.pth",
            str(upscale_factor)
        ]
        print(f"Running GFPGAN face restoration: {' '.join(enhance_cmd)}")
        res_enhance = subprocess.run(enhance_cmd, capture_output=True, text=True)
        
        print("\n=== GFPGAN STDOUT ===")
        print(res_enhance.stdout)
        print("=====================")
        
        if res_enhance.returncode != 0:
            print("GFPGAN Stderr:", res_enhance.stderr)
            raise RuntimeError(f"GFPGAN failed with exit code {res_enhance.returncode}")
            
        final_video_input = enhanced_output_path
        
    # ─── Stage 3: FFmpeg (Merge Audio) ───
    print("\n--- STAGE 3: Merging audio using FFmpeg ---")
    final_output_path = "/content/final_result.mp4"
    ffmpeg_cmd = [
        "ffmpeg", "-y",
        "-i", final_video_input,
        "-i", audio_path,
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "18",
        "-c:a", "aac",
        "-b:a", "192k",
        "-map", "0:v:0",
        "-map", "1:a:0",
        "-shortest",
        "-movflags", "+faststart",
        final_output_path
    ]
    print(f"Running FFmpeg: {' '.join(ffmpeg_cmd)}")
    res_ffmpeg = subprocess.run(ffmpeg_cmd, capture_output=True, text=True)
    if res_ffmpeg.returncode != 0:
        print("FFmpeg Stderr:", res_ffmpeg.stderr)
        raise RuntimeError("FFmpeg audio merging failed.")
        
    print(f"\n✅ Lip-sync complete! Video saved to: {final_output_path}")
    
    # Display the video inside Colab
    from IPython.display import HTML
    from base64 import b64encode
    mp4 = open(final_output_path,'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
    display(HTML(f'<video width=400 controls><source src="{data_url}" type="video/mp4"></video>'))
    
except Exception as e:
    print(f"❌ Error: {e}")
finally:
    shutil.rmtree(work_dir, ignore_errors=True)
